In [1]:
import json
from datasets import Dataset
import pandas as pd

# Asumsi Anda menyimpan data di file 'data.json' yang diunggah ke Kaggle
# (Ubah path file sesuai dengan direktori dataset Anda di Kaggle, misalnya '/kaggle/input/dataset-name/data.json')
file_path = '/kaggle/input/datasets/jauharulumam/qa-permenkes/dataset_qa_permenkes_astra final.json'

# Load data JSON
with open(file_path, 'r', encoding='utf-8') as f:
    raw_data = json.load(f)

# Siapkan list kosong untuk menampung format yang baru
formatted_data = []

# Ekstrak question dan answer menjadi format 'conversations'
for pasal_data in raw_data:
    for qa in pasal_data['qa_pairs']:
        # Format percakapan standar (user dan assistant)
        conversation = [
            {"role": "user", "content": qa["question"]},
            {"role": "assistant", "content": qa["answer"]}
        ]
        # Masukkan ke dalam dictionary dengan key 'conversations'
        formatted_data.append({"conversations": conversation})

# Konversi list dictionary ke Hugging Face Dataset
dataset = Dataset.from_list(formatted_data)

In [3]:
dataset[0]

{'conversations': [{'role': 'user', 'content': 'Sebutkan bunyi Pasal 1.'},
  {'role': 'assistant',
   'content': 'Pasal 1\n\nDalam Peraturan Menteri ini yang dimaksud dengan: 1. Jaringan Dokumentasi dan Informasi Hukum Nasional yang selanjutnya disingkat JDIHN adalah wadah pendayagunaan bersama atas dokumen hukum secara tertib, terpadu, dan berkesinambungan, serta merupakan sarana pemberian pelayanan informasi hukum secara lengkap, akurat, mudah, dan cepat. 2. Jaringan Dokumentasi dan Informasi Hukum di lingkungan Kementerian Kesehatan yang selanjutnya disebut JDIH Kemenkes adalah suatu sistem pengelolaan dan pendayagunaan bersama dokumen hukum dan informasi hukum di bidang kesehatan secara tertib, terpadu dan berkesinambungan serta merupakan sarana pemberian pelayanan informasi hukum secara lengkap, akurat, mudah dan cepat. 3. Dokumen Hukum adalah produk hukum yang berupa peraturan perundang-undangan atau produk hukum selain peraturan perundang-undangan yang meliputi namun tidak terba

In [14]:
import json
import os
import concurrent.futures
from datasets import Dataset
from openai import OpenAI

from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
openaiapi = user_secrets.get_secret("OPENAI_API_KEY")

client = OpenAI(api_key=openaiapi, timeout=900.0)

# Asumsikan 'original_dataset' adalah dataset Anda yang berisi 70 baris

system_prompt = """
Anda adalah asisten AI ahli bahasa. Tugas Anda memparafrasekan percakapan sebanyak 10 kali.
Pertahankan makna asli, kelengkapan informasi, dan konteks hukum/formal.
Output HARUS dalam format JSON.
Struktur JSON:
{
  "variations": [
    {"role_user": "...", "role_assistant": "..."}
  ]
}
"""

def process_single_row(item):
    index, row = item
    print(f"Mulai memproses baris ke-{index + 1}...")
    
    user_content = next(c['content'] for c in row['conversations'] if c['role'] == 'user')
    assistant_content = next(c['content'] for c in row['conversations'] if c['role'] == 'assistant')
    
    user_prompt = f"Tolong parafrase percakapan ini 10 kali.\n\nUser: {user_content}\nAssistant: {assistant_content}"
    
    try:
        response = client.chat.completions.create(
            model="gpt-5.6-luna",
            service_tier="flex",
            response_format={ "type": "json_object" },
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_prompt}
            ],
        )
        
        result_json = json.loads(response.choices[0].message.content)
        
        # Format hasil
        new_conversations = [row['conversations']] # Masukkan data asli
        for var in result_json['variations']:
            new_conversations.append([
                {'role': 'user', 'content': var['role_user']},
                {'role': 'assistant', 'content': var['role_assistant']}
            ])
        
        print(f"Baris ke-{index + 1} Selesai!")
        return new_conversations
        
    except Exception as e:
        print(f"Terjadi kesalahan pada baris {index + 1}: {e}")
        return [row['conversations']] # Jika error, kembalikan data aslinya saja

augmented_data = []

# Menggunakan max_workers=20 berarti memproses 20 data sekaligus secara paralel
with concurrent.futures.ThreadPoolExecutor(max_workers=20) as executor:
    # Siapkan data dengan indexnya
    items_to_process = [(i, row) for i, row in enumerate(dataset)]
    
    # Jalankan secara paralel
    results = executor.map(process_single_row, items_to_process)
    
    # Kumpulkan semua hasil
    for res in results:
        augmented_data.extend(res)

# Buat dataset baru
augmented_dataset = Dataset.from_dict({'conversations': augmented_data})
print(f"\nSelesai! Jumlah baris dataset sekarang: {augmented_dataset.num_rows}")

Mulai memproses baris ke-1...
Mulai memproses baris ke-2...
Mulai memproses baris ke-3...
Mulai memproses baris ke-4...
Mulai memproses baris ke-5...
Mulai memproses baris ke-6...
Mulai memproses baris ke-7...
Mulai memproses baris ke-8...
Mulai memproses baris ke-9...
Mulai memproses baris ke-10...
Mulai memproses baris ke-11...
Mulai memproses baris ke-12...
Mulai memproses baris ke-13...
Mulai memproses baris ke-14...
Mulai memproses baris ke-15...
Mulai memproses baris ke-16...
Mulai memproses baris ke-17...
Mulai memproses baris ke-18...
Mulai memproses baris ke-19...
Mulai memproses baris ke-20...
Baris ke-10 Selesai!
Mulai memproses baris ke-21...
Baris ke-6 Selesai!
Mulai memproses baris ke-22...
Baris ke-13 Selesai!
Mulai memproses baris ke-23...
Baris ke-17 Selesai!
Mulai memproses baris ke-24...
Baris ke-9 Selesai!
Mulai memproses baris ke-25...
Baris ke-8 Selesai!
Mulai memproses baris ke-26...
Baris ke-16 Selesai!
Mulai memproses baris ke-27...
Baris ke-12 Selesai!
Mulai m

In [15]:
augmented_dataset

Dataset({
    features: ['conversations'],
    num_rows: 776
})

In [16]:
import json

# Nama file tujuan
output_file = "QA_augmented_data.json"

# Buka file dan simpan data
# Parameter ensure_ascii=False sangat penting agar karakter bahasa Indonesia/spesial tidak menjadi simbol Unicode aneh
with open(output_file, "w", encoding="utf-8") as f:
    json.dump({'conversations': augmented_data}, f, ensure_ascii=False, indent=4)

print(f"Data berhasil disimpan ke {output_file}")

Data berhasil disimpan ke QA_augmented_data.json
